# Inbox Agent — Stage A

Deterministic triage over a frozen snapshot. The pipeline is:

**fetch → prefilter → classify → propose → interrupt (human review) → execute → learn**

Everything is dry-run by default (`INBOX_DRY_RUN` unset or falsey). No cell in this notebook mutates real Gmail state on a first run — `execute` simulates every action and logs it as `dry_run: true` in the audit trail. Corrections you make at the review step become durable preference rules for next time.

In [ ]:
# Cell 1 — wiring
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.types import Command

from inbox_agent.audit import AuditLog
from inbox_agent.config import (
    load_settings, get_llm, get_embeddings, mask, use_model, describe_models,
)
from inbox_agent.gmail import SnapshotGmailClient
from inbox_agent.graph import build_graph
from inbox_agent.policy import load_policy
from inbox_agent.render import render_review, review_table, respond, approve_all, audit_table
from inbox_agent.store import PreferenceStore, build_store
from inbox_agent.models import ReviewRequest, Action
import os, pandas as pd

settings = load_settings()
policy   = load_policy(settings)
print(f"backend    : {settings.backend}")
print(f"dry_run    : {settings.dry_run}")
print(f"policy     : {policy.version} ({policy.source})")
print(f"forbidden  : {sorted(settings.forbidden_actions)}")
print(f"langsmith  : {mask(os.getenv('LANGSMITH_API_KEY'))}")
print()
for r in describe_models():
    print(f"  {r['name']:<10} {r['cost']:<6} {r['backend']:<11} {r['model_id']}")


In [ ]:
# Cell 2 — components
client = SnapshotGmailClient(settings.snapshot_dir / "threads.json")
prefs  = PreferenceStore(build_store(get_embeddings()))
log    = AuditLog(settings.audit_log)
# Pick a model by name - no .env edit, no kernel restart.
# Run describe_models() below to see what is available and what it costs.
MODEL  = "gemma"          # "gemma" | "nemotron" | "4o-mini" | "glm" | "sonnet"
llm    = use_model(MODEL)

# SqliteSaver.from_conn_string is a context manager, not a plain constructor.
# We enter it manually (rather than `with ... as checkpointer:`) so the same
# connection stays open and usable across later notebook cells (Cell 3's
# graph.invoke and Cell 4's resume both need the same checkpointer). Verified
# against langgraph-checkpoint-sqlite 3.1.0: __enter__() returns a live
# SqliteSaver backed by an open sqlite3 connection. The trade-off is that we
# never call cm.__exit__() in a notebook session — the connection closes when
# the kernel process exits, which is fine for this lab notebook.
cm = SqliteSaver.from_conn_string("inbox_agent/checkpoints.sqlite")
checkpointer = cm.__enter__()

graph = build_graph(client=client, prefs=prefs, policy=policy, llm=llm,
                    settings=settings, log=log, checkpointer=checkpointer)
print(f"{len(client.list_threads(limit=500))} threads in snapshot")


In [ ]:
# Cell 3 — run to the review gate
config = {"configurable": {"thread_id": "session-1"}}
result = graph.invoke({"limit": settings.snapshot_size}, config)

request = ReviewRequest.model_validate(result["__interrupt__"][0].value)
pd.DataFrame(review_table(request))


In [ ]:
# Cell 4 — respond
# Approve everything except the threads you name. Add edits to teach it.
response = respond(
    request,
    reject=[],           # e.g. ["1a040e33a02ec514"]
    edit={},             # e.g. {"1a040d7d5d69e611": [Action(kind="label", thread_id="1a040d7d5d69e611", params={"label": "Finance"})]}
    instructions=[],
)
final = graph.invoke(Command(resume=response.model_dump(mode="json")), config)
print(f"executed : {len(final['executed'])} actions")
print(f"learned  : {len(final['learned'])} new rules")


In [ ]:
# Cell 5 — what it learned, and what it did
display(pd.DataFrame(prefs.as_table()))
display(pd.DataFrame(audit_table(log)))


## Next

Flip `INBOX_DRY_RUN=false` only after the proposals look right for several runs. Stage B (tool-calling agent) runs against this same snapshot and preference store.